# Traffic Sign Detection — Video Test (real application demo)

Runs the trained detector on a driving video, frame by frame, and shows the **current speed limit**
on screen like an in-car driver-assist display.

**Before running**
1. Put the weights from the training notebook in `weights/` (`best.onnx` or `best.pt`).
2. Put your driving video in `videos/` and set `VIDEO_PATH` below.

Runs locally on CPU (no GPU needed). ONNX is usually faster on CPU than `.pt`.

In [ ]:
import re
import time
from collections import Counter, deque
from pathlib import Path

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

VIDEO_PATH = ROOT / 'videos' / 'input.mp4'      # ← change to your video file
WEIGHTS = ROOT / 'weights' / 'best.onnx'        # or ROOT / 'weights' / 'best.pt'
OUT_PATH = ROOT / 'results' / f'{VIDEO_PATH.stem}_annotated.mp4'

CONF = 0.40          # minimum confidence to accept a detection
VOTE_WINDOW = 5      # recent speed-sign sightings used to decide the current limit
MAX_FRAMES = None    # e.g. 600 to test only the first ~20 s

for p in (VIDEO_PATH, WEIGHTS):
    assert p.exists(), f'missing {p}'
OUT_PATH.parent.mkdir(exist_ok=True)

## Speed-limit logic
The detector finds signs in each frame; this small tracker turns those detections into one
stable "current speed limit". It keeps the last few speed-sign sightings and shows the most common
value, so a single wrong frame does not make the display flicker. The limit is kept until a new
speed sign is seen (just like a real road).

In [ ]:
class SpeedLimitTracker:
    def __init__(self, window=VOTE_WINDOW):
        self.recent = deque(maxlen=window)

    def update(self, labels):
        for label in labels:
            m = re.search(r'(\d+)', label) if 'speed' in label.lower() else None
            if m:
                self.recent.append(int(m.group(1)))
        return Counter(self.recent).most_common(1)[0][0] if self.recent else None

# self-check
t = SpeedLimitTracker(window=3)
assert t.update([]) is None
assert t.update(['Speed Limit 60', 'Stop']) == 60
assert t.update(['Red Light']) == 60                           # held with no new sign
assert t.update(['Speed Limit 80']) in (60, 80)
assert t.update(['Speed Limit 80']) == 80                      # majority switches

def draw_limit(frame, limit):
    """Draw a round speed-limit sign in the top-left corner."""
    c, r = (70, 70), 50
    cv2.circle(frame, c, r, (255, 255, 255), -1)
    cv2.circle(frame, c, r, (0, 0, 220), 9)
    text = str(limit) if limit else '--'
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, 1.3, 3)
    cv2.putText(frame, text, (c[0] - tw // 2, c[1] + th // 2), cv2.FONT_HERSHEY_DUPLEX, 1.3, (0, 0, 0), 3)
    return frame

## Run on the video

In [ ]:
model = YOLO(str(WEIGHTS), task='detect')
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer = cv2.VideoWriter(str(OUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
print(f'{VIDEO_PATH.name}: {w}x{h} @ {fps:.0f} fps, {total} frames')

tracker, log, keyframes = SpeedLimitTracker(), [], []
stride = max(total // 16, 1)   # sample frames spread over the whole video
start = time.perf_counter()
i = 0
while MAX_FRAMES is None or i < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    r = model.predict(frame, conf=CONF, imgsz=640, verbose=False)[0]
    labels = [model.names[int(c)] for c in r.boxes.cls]
    limit = tracker.update(labels)
    out = draw_limit(r.plot(), limit)
    writer.write(out)
    log.append({'frame': i, 'time_s': i / fps, 'ms': sum(r.speed.values()), 'n_signs': len(labels),
                'labels': labels, 'conf': [float(c) for c in r.boxes.conf], 'limit': limit})
    if labels and i % stride == 0 and len(keyframes) < 8:
        keyframes.append((i, cv2.cvtColor(out, cv2.COLOR_BGR2RGB)))
    if i % 100 == 0:
        print(f'frame {i}/{total}  current limit: {limit}')
    i += 1
cap.release(); writer.release()
wall = time.perf_counter() - start
print(f'saved {OUT_PATH}  (open with QuickTime / VLC)')

## Performance on the video
There are no ground-truth labels for your own video, so this is a **qualitative + speed** check:
processing speed, how often signs are detected, which classes appear, their confidence,
and how the displayed speed limit changes over time. Watch the annotated video to judge correctness.

In [ ]:
df = pd.DataFrame(log)
all_labels = [l for ls in df['labels'] for l in ls]
all_conf = [c for cs in df['conf'] for c in cs]
video_summary = pd.DataFrame({
    'metric': ['frames processed', 'video length (s)', 'avg model time (ms/frame)', 'processing FPS (end-to-end)',
               'real-time?', 'frames with ≥1 sign', 'total detections', 'mean confidence', 'speed limits shown'],
    'value': [len(df), round(len(df) / fps, 1), round(df['ms'].mean(), 1), round(len(df) / wall, 1),
              'yes' if len(df) / wall >= fps else f'no (video is {fps:.0f} fps)',
              f"{(df['n_signs'] > 0).mean():.1%}", len(all_labels),
              round(sum(all_conf) / len(all_conf), 3) if all_conf else '-',
              ', '.join(map(str, sorted(df['limit'].dropna().astype(int).unique()))) or '-'],
})
display(video_summary)

fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
if all_labels:
    pd.Series(all_labels).value_counts().plot.barh(ax=ax[0])
ax[0].set_title('Detections per class')
ax[1].hist(all_conf, bins=20, range=(CONF, 1)); ax[1].set_title('Detection confidence'); ax[1].set_xlabel('confidence')
ax[2].step(df['time_s'], df['limit'], where='post'); ax[2].set_title('Displayed speed limit over time')
ax[2].set_xlabel('time (s)'); ax[2].set_ylabel('km/h')
plt.tight_layout(); plt.savefig(ROOT / 'results' / f'{VIDEO_PATH.stem}_stats.png', dpi=120); plt.show()

In [ ]:
if keyframes:
    cols = 4
    rows = (len(keyframes) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (idx, img) in zip(axes.flat, keyframes):
        ax.imshow(img); ax.set_title(f'frame {idx} ({idx / fps:.1f}s)')
    plt.tight_layout(); plt.savefig(ROOT / 'results' / f'{VIDEO_PATH.stem}_frames.png', dpi=120); plt.show()
else:
    print('No signs detected — try lowering CONF, or check that the signs in the video look like the training data.')